In [0]:
from pyspark.sql import functions as F

root  = "data_analytics.silverlayer"
gold_root = "data_analytics.goldlayer"

#Creating dimention views 

## Customers dimention

In [0]:
df1 = spark.table(f"{root}.customer_information")
df2 = spark.table(f"{root}.customer_demographic_info")
df3 = spark.table(f"{root}.customer_location")

df1 = (df1.join(df2, df1.Customer_key == df2.Customer_key, "left")
       .join(df3, df1.Customer_key == df3.Customer_key, "left")
       .select(
               df1.Customer_id,
               df1.Customer_key,
               df1.First_name,
               df1.Last_name,
               df1.Marital_status,
               F.when( (df1.Gender =="n/a"), F.coalesce(df2.Gender, F.lit("n/a")))
               .otherwise(df1.Gender).alias("Gender"),
               df3.Country,
               df2.Birth_date,
               df1.Date_created
               )
       .distinct()
       )
df1.write.mode("overwrite").saveAsTable(f"{gold_root}.dim_customer_infomation")

##Products dimention

In [0]:
df1 = spark.table(f"{root}.product_information")
df2 = spark.table(f"{root}.product_catagory")

df1 = (df1.join(df2, df1.Product_category == df2.Product_category, "left")
       .select(df1.Product_id,
               df1.Product_category,
               df1.Product_name,
               df1.Product_description,

# Creating the fact table